In [1]:
platformID = 'INS'

In [2]:
from datetime import datetime
import pandas as pd
pd.set_option('display.float_format', '{:.2f}'.format)
import numpy as np
import os 

import psycopg2


## import helper 

In [3]:
import sys
from pathlib import Path

try:
    # Works in Python scripts
    helper_path = Path(__file__).resolve().parent.parent / "helper"
except NameError:
    # Works in Jupyter notebooks
    helper_path = Path().resolve().parent / "helper"

sys.path.insert(0, str(helper_path))

# Now import your modules 
from config import gam_info
import test_functions 
from functions import lookup_loader, calculate_rolling_avg_country_split

In [4]:
lookup = lookup_loader(gam_info, platformID, '3',
                       with_country=True, country_col=['YT-_FBE_codes'],
                      with_pop_col=True)
week_tester = lookup['week_tester']
socialmedia_accounts = lookup['socialmedia_accounts']
country_codes = lookup['country_codes']


'''# country
country_codes = pd.read_excel(f"../../{gam_info['lookup_file']}", sheet_name='CountryID', 
                              keep_default_na=False)

# week 
week_tester = pd.read_excel(f"../../{gam_info['lookup_file']}", sheet_name='GAM Period')
week_tester['w/c'] = pd.to_datetime(week_tester['w/c'])

# social media accounts
socialmedia_accounts = pd.read_excel("../helper/ins_account_lookup.xlsx")

### RUN TESTS
test_functions.test_lookup_files(country_codes, ['PlaceID'], [f"{platformID}_3_0", f"{platformID}_3_1", f"{platformID}_3_2"], 
                                 test_step="lookup files - ensuring country codes is correct")

test_functions.test_lookup_files(week_tester, ['w/c'], [f"{platformID}_3_3", f"{platformID}_3_4", f"{platformID}_3_5"], 
                                 test_step = "lookup files - ensuring week tester is correct")

test_functions.test_lookup_files(socialmedia_accounts, ['Channel ID'], [f"{platformID}_3_6", f"{platformID}_3_7", f"{platformID}_3_8"], 
                                 test_step = "lookup files - ensuring social media accounts is correct")

'''

✅ Test INS_3_00 passed: lookup DataFrame is not empty.
...updating logbook...

✅ Test INS_3_01 passed: No combinations occurs more than once a week.
...updating logbook...

✅ Test INS_3_02 passed: No missing values in lookup.
...updating logbook...

✅ Test INS_3_03 passed: lookup DataFrame is not empty.
...updating logbook...

✅ Test INS_3_04 passed: No combinations occurs more than once a week.
...updating logbook...

✅ Test INS_3_05 passed: No missing values in lookup.
...updating logbook...

✅ Test INS_3_06 passed: lookup DataFrame is not empty.
...updating logbook...

✅ Test INS_3_07 passed: No combinations occurs more than once a week.
...updating logbook...

✅ Test INS_3_08 passed: No missing values in lookup.
...updating logbook...



'# country\ncountry_codes = pd.read_excel(f"../../{gam_info[\'lookup_file\']}", sheet_name=\'CountryID\', \n                              keep_default_na=False)\n\n# week \nweek_tester = pd.read_excel(f"../../{gam_info[\'lookup_file\']}", sheet_name=\'GAM Period\')\nweek_tester[\'w/c\'] = pd.to_datetime(week_tester[\'w/c\'])\n\n#\xa0social media accounts\nsocialmedia_accounts = pd.read_excel("../helper/ins_account_lookup.xlsx")\n\n### RUN TESTS\ntest_functions.test_lookup_files(country_codes, [\'PlaceID\'], [f"{platformID}_3_0", f"{platformID}_3_1", f"{platformID}_3_2"], \n                                 test_step="lookup files - ensuring country codes is correct")\n\ntest_functions.test_lookup_files(week_tester, [\'w/c\'], [f"{platformID}_3_3", f"{platformID}_3_4", f"{platformID}_3_5"], \n                                 test_step = "lookup files - ensuring week tester is correct")\n\ntest_functions.test_lookup_files(socialmedia_accounts, [\'Channel ID\'], [f"{platformID}_3_6", f"{pl

# ingest 

In [5]:
engagements = pd.read_csv(f"../data/processed/{platformID}/{gam_info['file_timeinfo']}_{platformID}_engagements_final.csv",)
                                             
engagements['w/c'] = pd.to_datetime(engagements['w/c'])
engagements['Channel ID'] = engagements['Channel ID'].dropna()
engagements = engagements.dropna(subset=['ServiceID'])
engagements.sort_values(by='w/c')['w/c'].unique()

<DatetimeArray>
['2025-11-10 00:00:00', '2025-11-17 00:00:00', '2025-11-24 00:00:00',
 '2025-12-01 00:00:00', '2025-12-08 00:00:00', '2025-12-15 00:00:00',
 '2025-12-22 00:00:00']
Length: 7, dtype: datetime64[ns]

In [6]:
country = pd.read_csv(f"../data/processed/{platformID}/{gam_info['file_timeinfo']}_{platformID}_REDSHIFT_geog.csv",
                     keep_default_na=False)
country['w/c'] = pd.to_datetime(country['w/c'])
country['PlatformID'] = platformID
country['Channel ID'] = country['Channel ID'].dropna()
country.sample()

,Channel ID,Channel Name,w/c,PlaceID,country_%,PlatformID
439,INS17841473849472688,BBC News Polska,2025-07-28,SPA,0.01,INS


In [7]:
country_annual_avg = calculate_rolling_avg_country_split(country, 'country_%',
                                                         country['w/c'].min(), country['w/c'].max())


# combine 

In [8]:
combined = engagements.merge(country, on=["Channel ID", "w/c"], how='left', indicator=True)
print(f"Initial merge mismatches:\n{combined._merge.value_counts()}")
combined_inner = combined[combined['_merge'] == 'both'].drop(columns='_merge')
combined_left = combined[combined['_merge'] == 'left_only'].drop(columns='_merge')

left_matched = combined_left[engagements.columns].merge(country_annual_avg, on=["Channel ID", "w/c"], 
                                                        how='left', indicator=True)
print(f"Country second merge mismatches:\n{left_matched._merge.value_counts()}")
'''cols_to_clean = ['Channel Name', 'PlaceID', 'country_%',]
for col in cols_to_clean:
    left_matched[f"{col}"] = left_matched[f"{col}_x"].fillna(left_matched[f"{col}_y"])
    left_matched = left_matched.drop(columns=[f"{col}_x", f"{col}_y"])

temp = left_matched[left_matched['_merge'] == 'left_only'].drop(columns='_merge')
temp = temp.merge(socialmedia_accounts[['Channel ID', 'IG Handle']], on='Channel ID', how='left')
missing_country_perc = pd.read_excel("../data/stale/missing ig countries.xlsx")
missing_country_perc['country code'] = missing_country_perc['country code'].str.upper()
missing_country_perc.rename(columns={'country code': 'PlaceID', 'Total': 'country_%'})
temp = temp.merge(missing_country_perc, on='IG Handle', how='inner')
'''

cols = ['Channel ID', 'Channel Name', 
        'w/c', 'ServiceID',
        'plays', 'impressions',
        '30 view', 'IG Modelled Factor', 
        'PlaceID', 'engaged_reach', 'country_%']
temp = pd.concat([combined_inner, left_matched])[cols]

Initial merge mismatches:
_merge
both          18539
left_only         0
right_only        0
Name: count, dtype: int64
Country second merge mismatches:
_merge
left_only     0
right_only    0
both          0
Name: count, dtype: int64


In [9]:

cols = ['Channel ID', 'Channel Name', 
       'w/c', 'ServiceID',  'plays',  'PlaceID', 'engaged_reach', 'country_%']
engagement_country = pd.concat([combined_inner, temp])[cols].rename(columns={'IG Engaged Persian Exception': 'IG Engaged Users'})



In [10]:
to_clean_country = country_codes[['PlaceID', 'YT-_FBE_codes', gam_info['population_column']]]
clean_country = engagement_country.merge(to_clean_country, on='PlaceID', how='left', indicator=True)
print(clean_country.shape)
final_ig_data = clean_country.drop_duplicates(subset=['PlaceID', 'w/c', gam_info['population_column'],
                                                      'Channel ID', 'Channel Name'])
print(final_ig_data.shape)
final_ig_data['engaged_reach'] = final_ig_data['engaged_reach'].fillna(0)
final_ig_data['country_%'] = final_ig_data['country_%'].fillna(0)
final_ig_data['uv_by_country'] = final_ig_data['engaged_reach'] * final_ig_data['country_%']

(37078, 11)
(18539, 11)


/var/folders/wl/kjzhstq972q0cn5zv7h_jfbw0000gn/T/ipykernel_94690/1606694705.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  final_ig_data['engaged_reach'] = final_ig_data['engaged_reach'].fillna(0)
/var/folders/wl/kjzhstq972q0cn5zv7h_jfbw0000gn/T/ipykernel_94690/1606694705.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  final_ig_data['country_%'] = final_ig_data['country_%'].fillna(0)
/var/folders/wl/kjzhstq972q0cn5zv7h_jfbw0000gn/T/ipykernel_94690/1606694705.py:9: SettingWithCopyWarning: 
A value i

# store 

In [11]:
print(final_ig_data.shape)
final_ig_data = final_ig_data.dropna(subset='uv_by_country')
print(final_ig_data.shape)

cols = ['w/c', 'PlaceID', 'ServiceID', 'Channel ID', 'uv_by_country']
final_ig_data[cols].to_csv(f"../data/processed/{platformID}/{gam_info['file_timeinfo']}_{platformID}_uniqueViewer_country.csv",
                     index=None)


(18539, 12)
(18539, 12)
